In [4]:
pip install xgboost pytorch-tabnet 

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\chaka\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [5]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# XGBoost
from xgboost import XGBClassifier

# TabNet
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [6]:
#data collection
df = pd.read_csv(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\Datasets\rideflow_datasets.csv")

#feature engineering
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month

# Time bin
df['time_bin'] = df['timestamp'].dt.floor('30min')

# Demand per time bin
demand = df.groupby('time_bin').size().reset_index(name='demand')

df = df.merge(demand, on='time_bin')

#addtional features
df['is_weekend'] = df['day'].isin([5,6]).astype(int)
df['peak_hour'] = df['hour'].isin([8,9,17,18,19]).astype(int)

# Spatial clustering (IMPORTANT 🔥)
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=20, random_state=42)
df['zone'] = kmeans.fit_predict(df[['pickup_lat','pickup_long']])


In [7]:
#label creation
low = df['demand'].quantile(0.33)
high = df['demand'].quantile(0.66)

def label(x):
    if x >= high:
        return 2  # high
    elif x >= low:
        return 1  # medium
    else:
        return 0  # low

df['label'] = df['demand'].apply(label)

In [8]:
#features and target

features = [
    'pickup_lat',
    'pickup_long',
    'hour',
    'day',
    'month',
    'demand',
    'is_weekend',
    'peak_hour',
    'zone'
]

X = df[features]
y = df['label']

In [9]:
#traib test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

XGBOOST - baseline

In [10]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)

# Evaluation
xgb_preds = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_preds))
print(classification_report(y_test, xgb_preds))

XGBoost Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2908
           1       1.00      1.00      1.00      3472
           2       1.00      1.00      1.00      3620

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [11]:
joblib.dump(xgb_model, "xgb_hotspot.pkl")
joblib.dump(kmeans, "kmeans.pkl")

['kmeans.pkl']

TABNET MODEL (ADVANCED)

In [12]:
tabnet_model = TabNetClassifier(
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    verbose=1
)

tabnet_model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_test.values, y_test.values)],
    eval_metric=['accuracy'],
    max_epochs=50,
    patience=10,
    batch_size=1024
)

# Evaluation
tab_preds = tabnet_model.predict(X_test.values)

print("TabNet Accuracy:", accuracy_score(y_test, tab_preds))
print(classification_report(y_test, tab_preds))

c:\Users\chaka\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.03349 | val_0_accuracy: 0.3812  |  0:00:07s
epoch 1  | loss: 0.3309  | val_0_accuracy: 0.4477  |  0:00:13s
epoch 2  | loss: 0.16471 | val_0_accuracy: 0.3779  |  0:00:20s
epoch 3  | loss: 0.12586 | val_0_accuracy: 0.4165  |  0:00:26s
epoch 4  | loss: 0.10805 | val_0_accuracy: 0.3995  |  0:00:32s
epoch 5  | loss: 0.09627 | val_0_accuracy: 0.481   |  0:00:39s
epoch 6  | loss: 0.10271 | val_0_accuracy: 0.6762  |  0:00:46s
epoch 7  | loss: 0.08492 | val_0_accuracy: 0.6035  |  0:00:52s
epoch 8  | loss: 0.08291 | val_0_accuracy: 0.6066  |  0:00:59s
epoch 9  | loss: 0.07517 | val_0_accuracy: 0.5825  |  0:01:06s
epoch 10 | loss: 0.07678 | val_0_accuracy: 0.7478  |  0:01:13s
epoch 11 | loss: 0.06933 | val_0_accuracy: 0.7682  |  0:01:20s
epoch 12 | loss: 0.06344 | val_0_accuracy: 0.8564  |  0:01:27s
epoch 13 | loss: 0.06725 | val_0_accuracy: 0.9067  |  0:01:34s
epoch 14 | loss: 0.07655 | val_0_accuracy: 0.9543  |  0:01:42s
epoch 15 | loss: 0.06646 | val_0_accuracy: 0.9771  |  0

c:\Users\chaka\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


TabNet Accuracy: 0.9995
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2908
           1       1.00      1.00      1.00      3472
           2       1.00      1.00      1.00      3620

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [14]:
tabnet_model.save_model(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\saved_models\tabnet_hotspot.zip")

Successfully saved model at C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\saved_models\tabnet_hotspot.zip.zip


'C:\\Users\\chaka\\Preethu\\My_Git_Repo\\Final Project_Rideflow\\saved_models\\tabnet_hotspot.zip.zip'

In [15]:
#predition funtion for streamlit
def predict_hotspot(input_dict, model, kmeans):
    df = pd.DataFrame([input_dict])
    
    # Create derived features
    df['is_weekend'] = int(df['day'].iloc[0] in [5,6])
    df['peak_hour'] = int(df['hour'].iloc[0] in [8,9,17,18,19])
    
    df['zone'] = kmeans.predict(df[['pickup_lat','pickup_long']])
    
    X = df[[
        'pickup_lat','pickup_long','hour','day',
        'month','demand','is_weekend','peak_hour','zone'
    ]]
    
    pred = model.predict(X)
    
    mapping = {0: "Low", 1: "Medium", 2: "High"}
    return mapping[int(pred[0])]

In [16]:
#testing
sample = {
    'pickup_lat': 13.0827,
    'pickup_long': 80.2707,
    'hour': 18,
    'day': 4,
    'month': 3,
    'demand': 120
}

# Load model
model = joblib.load("xgb_hotspot.pkl")
kmeans = joblib.load("kmeans.pkl")

print(predict_hotspot(sample, model, kmeans))

High
